# Inferensi IndoBERT Pantun (Colab)

Notebook ini dipakai untuk menggunakan model IndoBERT token-classification yang sudah disimpan di Google Drive.

In [ ]:
!pip -q install -U "pyarrow>=19,<24" transformers pandas

In [ ]:
from google.colab import drive
from pathlib import Path
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification

drive.mount('/content/drive')

WORKSPACE_DIR = Path('/content/drive/MyDrive/NERPantun')
MODEL_DIR = WORKSPACE_DIR / 'models/indobert-pantun-ner'

if not MODEL_DIR.exists():
    raise FileNotFoundError(f'Model tidak ditemukan: {MODEL_DIR}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Model dir:', MODEL_DIR)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR))
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR))
model.to(device)
model.eval()

id2label = model.config.id2label
label2id = model.config.label2id
print('Labels:', id2label)

In [ ]:
def predict_tokens(tokens, max_length=128):
    enc = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=max_length,
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits

    pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()
    word_ids = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
    ).word_ids()

    results = []
    prev = None
    for i, widx in enumerate(word_ids):
        if widx is None or widx == prev:
            prev = widx
            continue
        results.append((tokens[widx], id2label[int(pred_ids[i])]))
        prev = widx
    return results

def predict_text(text):
    tokens = text.strip().split()
    return predict_tokens(tokens)

In [ ]:
# Contoh prediksi 1
sample_tokens = ['ke', 'spbu', 'membeli', 'bensin', 'aku', 'gak', 'tau', 'kenapa']
print(predict_tokens(sample_tokens))

# Contoh prediksi 2 (teks)
sample_text = 'ke spbu membeli bensin aku gak tau kenapa'
print(predict_text(sample_text))

## (Opsional) Prediksi batch dari CSV

CSV minimal punya kolom `text` (satu pantun per baris). Hasil disimpan ke file baru di Drive.

In [ ]:
INPUT_CSV = WORKSPACE_DIR / 'inference/input_pantun.csv'
OUTPUT_CSV = WORKSPACE_DIR / 'inference/output_pantun_with_tags.csv'

if INPUT_CSV.exists():
    df = pd.read_csv(INPUT_CSV)
    if 'text' not in df.columns:
        raise ValueError('CSV harus punya kolom: text')

    preds = []
    for txt in df['text'].fillna(''):
        tagged = predict_text(str(txt))
        tokens = [t for t, _ in tagged]
        labels = [l for _, l in tagged]
        preds.append({
            'text': txt,
            'tokens': ' | '.join(tokens),
            'pred_tags': ' | '.join(labels),
        })

    out_df = pd.DataFrame(preds)
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(OUTPUT_CSV, index=False)
    print('Saved:', OUTPUT_CSV)
    print(out_df.head(3))
else:
    print('Lewati batch CSV (file input belum ada):', INPUT_CSV)